# COMP4124 - Big Data Learning & Technologies<br>Group 5 - Project 1: Clustering of Large-Scale GPS Trajectory Data

**Dataset:** Microsoft GeoLife GPS Trajectories v1.3  
**Source:** [Click Here](https://www.microsoft.com/en-us/download/details.aspx?id=52367) or https://www.microsoft.com/en-us/download/details.aspx?id=52367  
**User Guide:** GeoLife User Guide v1.3 (2012/08/01)
<br><br>
**This entire prtoject has been uploaded on GitHub.** [Click Here](https://github.com/Abhinav-Tadi/Big_Data_Group_5) to view the repository.<br><br>

<font color='red'>Please read these instructions before running the Codebase File.<br></font>
After downloading the original dataset, please ensure that you unzip the zip file and place the unzipped folder and this Codebase file into a single folder. <br>
For Example, 
```
Group_5_Project/
├── Geolife Trajectories 1.3/
│   ├── Data/
│   └── User Guide-1.3.pdf
└── Group_5_Codebase.ipynb/
```
Alternatively, you can also download the Clean Dataset from [GitHub](https://github.com/Abhinav-Tadi/Big_Data_Group_5) and put the Clean Dataset and this Codebase file in a singular folder.
```
Group_5_Project/
├── Clean_Dataset/
└── Group_5_Codebase.ipynb/
```
<br>
If you chose to do this, you would not need to download the original dataset, nor run the Data Pre-Processing part of this codebase.

---
## Codebase Structure
This Codebase can be divided into 5 Sections.  
- Section 1: Setup
- Section 2: Data Pre-Processing
- Section 3: Common Tools
- Section 4: Methodologies 
- Section 5: End

---
## Section 1: Setup
---

### Importation

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    DoubleType, StringType
)
from pyspark.sql.window import Window
from pyspark.sql.functions import udf
import os, math, random

### Start Spark Session

<font color='red'>Note:</font> We are using 16 GB of RAM (Unified Memory). This is possible because the device on which the final experiment is run has 24 GB of Unified Memory. If your device does not have this much memory to allocate, please make 2 changes to the following code. 16g -> 8g (8 GB).  
```.config("spark.driver.memory", "16g")``` to ```.config("spark.driver.memory", "8g")```  
```.config("spark.executor.memory", "16g")``` to ```.config("spark.executor.memory", "8g")```

In [2]:
spark = SparkSession.builder \
    .appName("Group_5_Codebase") \
    .config("spark.sql.shuffle.partitions", "800") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 22:15:10 WARN Utils: Your hostname, Abhinavs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.132.196.51 instead (on interface en0)
26/03/29 22:15:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 22:15:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


### PATH Config

In [3]:
# Get the PATH of the folder in which the Codebase is located
PATH = os.getcwd() + "/"
print(f"PATH: {PATH}")

# Set the Raw Dataset Path
DATA_ROOT = PATH + "Geolife Trajectories 1.3/Data" 
# Set the Raw Dataset Path
OUTPUT_DIR = PATH + "Clean_Dataset"
# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH: /Users/abhinavtadi/Desktop/University of Nottingham/Sem 2/Big Data Learning and Technologies/Big_Data_Group_5/


---
## Section 2: Data Pre-Processing
---

The Data Pre-Processing section can be divided into 7 steps.
- Step 1: Parsing the .plt Files
- Step 2: Type Casting & Unit Conversion
- Step 3: Data Quality Checks
- Step 4: Cleaning the Pipeline
- Step 5: Feature Engineering
- Step 6: Transport Label Joining
- Step 7: Outputs

### <u>Additional Info about the Dataset</u>
### Dataset Facts (from User Guide v1.3)
| Property | Value |
|---|---|
| Users | 182 |
| Trajectories | 18,670 |
| GPS Points | 24,876,978 |
| Total distance | 1,292,951 km |
| Total duration | 50,176 hours |
| Dense sampling (1-5s or 5-10m) | 91.5% of trajectories |
| Users with transport labels | 73 |
| Collection period | April 2007 - August 2012 |
| Primary location | Beijing, China (also 30+ other cities) |
| All timestamps | GMT |

### .plt File Format (User Guide Section 4.1)
Lines 1-6 are a useless header - skip entirely.
From line 7 onwards, each line is one GPS point:
```
Field 1: Latitude  (decimal degrees)
Field 2: Longitude (decimal degrees)
Field 3: Always 0  (unused, per User Guide)
Field 4: Altitude in feet (-777 = invalid/missing)
Field 5: Days since 30/12/1899 (fractional) - equivalent to Fields 6+7
Field 6: Date string YYYY-MM-DD
Field 7: Time string HH:MM:SS
```
Example: `39.906631,116.385564,0,492,40097.5864583333,2009-10-11,14:04:30`

### labels.txt Format (User Guide Section 4.2)
Tab-separated. First line is a header row to skip.
Timestamp format uses slashes: `yyyy/MM/dd HH:mm:ss`
```
Start Time\tEnd Time\tTransportation Mode
2008/04/02 11:24:21\t2008/04/02 11:50:45\tbus
```
Valid modes: walk, bike, bus, car, taxi, subway, train, airplane, boat, run, motorcycle

**Label normalisation notes from User Guide:**
- `taxi` and `car` are equivalent - both mapped to `driving`
- `train` / `subway` may overlap due to Beijing light rail connectivity

### Folder Structure
```
Data/
├── 000/
│   ├── labels.txt   (only present for 73 of 182 users)
│   └── Trajectory/
│       ├── 20070804033032.plt
│       └── ...
├── 001/
...
└── 181/
```
---

## Step 1: Parsing the .plt Files

Before we can start pre-processing data, we need to first set some developmental constraints.

In [4]:
# Development subset flag.
# Per project brief, all subset usage must be documented.
# Set False only when running final experiments on the full dataset.
USE_SUBSET   = False
SUBSET_USERS = 10     # number of users to sample for development
RANDOM_SEED  = 42     # fixed seed ensures reproducibility

# Minimum GPS points for a trajectory to be kept after cleaning
MIN_POINTS = 10

# Speed artefact threshold - Justification in Section 4: Cleaning the Pipeline
MAX_SPEED_MPS = 300.0

# Beijing bounding box derived from User Guide Figure 1
# The majority of data is in Beijing; setting this True restricts to the
# geographically dense region but excludes other Chinese cities and overseas.
FILTER_TO_BEIJING = False
BEIJING_LAT_MIN, BEIJING_LAT_MAX =  39.4,  41.1
BEIJING_LON_MIN, BEIJING_LON_MAX = 115.4, 117.6

print(f"Data root      : {DATA_ROOT}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Subset mode    : {USE_SUBSET}" + (f" ({SUBSET_USERS} users, seed={RANDOM_SEED})" if USE_SUBSET else " -- FULL DATASET"))
print(f"Beijing filter : {FILTER_TO_BEIJING}")

# Sampling is done at the User level, not the point level, so that each
# sampled user contributes all their complete trajectories. This is essential
# because TRACLUS and K-medoids operate on whole trajectories.

all_users = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])
print(f"Users found in dataset: {len(all_users)}")
if len(all_users) != 182:
    print(f"WARNING: User Guide v1.3 states 182 users, found {len(all_users)}")

if USE_SUBSET:
    random.seed(RANDOM_SEED)
    selected_users = sorted(random.sample(all_users, min(SUBSET_USERS, len(all_users))))
    print(f"Subset Selected (seed={RANDOM_SEED}): {selected_users}")
else:
    selected_users = all_users
    print("Using All Users (Full Dataset)")

glob_patterns = [
    f"{DATA_ROOT}/{user}/Trajectory/*.plt"
    for user in selected_users
]

Data root      : /Users/abhinavtadi/Desktop/University of Nottingham/Sem 2/Big Data Learning and Technologies/Big_Data_Group_5/Geolife Trajectories 1.3/Data
Output dir     : /Users/abhinavtadi/Desktop/University of Nottingham/Sem 2/Big Data Learning and Technologies/Big_Data_Group_5/Clean_Dataset
Subset mode    : False -- FULL DATASET
Beijing filter : False
Users found in dataset: 182
Using All Users (Full Dataset)


Now, we begin loading .plt files. <br><br>
All .plt files are read as plain text in a single distributed Spark read.
`input_file_name()` gives us the source path of every line, from which we
extract `user_id` and `trajectory_id` via regex in the next step.
This avoids any driver-side looping over files.

In [5]:
raw_df = spark.read.text(glob_patterns)
raw_df = raw_df.withColumn("file_path", F.input_file_name())

total_raw_lines = raw_df.count()
print(f"Total raw lines (including headers): {total_raw_lines:,}")
raw_df.show(8, truncate=False)

Total raw lines (including headers): 24,988,998
+---------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                          |file_path                                                                                                                                                                                             |
+---------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Geolife trajectory                                             |file:///Users/abhinavtadi/Desktop/University%20of%20Nottingham/Sem%202/Big%20Data%20Learning

Path pattern: `.../Data/{user_id}/Trajectory/{trajectory_id}.plt`
Per the User Guide, each .plt file is named by its trajectory start time,
e.g. `20090811045202.plt` - we use the filename as a unique trajectory ID.

In [6]:
raw_df = raw_df \
    .withColumn(
        "user_id",
        F.regexp_extract("file_path", r"/Data/([^/]+)/Trajectory/", 1)
    ) \
    .withColumn(
        "trajectory_id",
        F.regexp_extract("file_path", r"/Trajectory/([^/]+)\.plt", 1)
    )

raw_df.select("user_id", "trajectory_id", "value").show(5, truncate=False)

+-------+--------------+------------------------------+
|user_id|trajectory_id |value                         |
+-------+--------------+------------------------------+
|010    |20081219114010|Geolife trajectory            |
|010    |20081219114010|WGS 84                        |
|010    |20081219114010|Altitude is in Feet           |
|010    |20081219114010|Reserved 3                    |
|010    |20081219114010|0,2,255,My Track,0,0,2,8421376|
+-------+--------------+------------------------------+
only showing top 5 rows


User Guide Section 4.1: lines 1-6 are useless headers, skip them.
GPS data lines always begin with a latitude value (a digit).

We use Fields 6+7 (date and time strings) for the timestamp rather than
Field 5 (days since 1899-12-30). The User Guide confirms they are equivalent
and the string representation is safer to parse in PySpark.

Field 3 is always 0 (User Guide confirmed) - discarded.
Field 4 altitude: -777 is the explicit invalid sentinel per User Guide.

In [7]:
# Two classes of bad lines exist in the raw data:
# 1. Header/text lines (e.g. 'My Track') -> safe_double() returns null lat -> dropped
# 2. Short lines with fewer than 7 comma-separated fields (blank lines, partial rows)
#    -> getItem(1), getItem(3) etc. throw INVALID_ARRAY_INDEX before safe_double runs
#
# Fix: pre-filter to only lines that contain at least 6 commas (= 7 fields),
# which is the exact structure of a valid GPS data line per User Guide s4.1.
# This runs before any array access and eliminates both classes of bad lines.

def safe_double(col_expr):
    # Cast to Double only if the value looks like a number, else null.
    return F.when(
        col_expr.rlike(r'^-?[0-9]+\.?[0-9]*$'),
        col_expr.cast(DoubleType())
    ).otherwise(None)

# Keep only lines with exactly 7 comma-separated fields (6 commas)
# A valid GeoLife GPS line: lat,lon,0,alt,days,date,time
valid_lines_df = raw_df.filter(
    F.size(F.split(F.col("value"), ",")) == 7
)

split_col = F.split(F.col("value"), ",")

data_df = valid_lines_df.select(
    F.col("user_id"),
    F.col("trajectory_id"),
    safe_double(split_col.getItem(0)).alias("latitude"),
    safe_double(split_col.getItem(1)).alias("longitude"),
    # Field 3 always 0 - skipped per User Guide s4.1
    safe_double(split_col.getItem(3)).alias("altitude_ft"),
    # Field 5 (days since 1899) skipped - using Fields 6+7 (User Guide s4.1)
    F.concat_ws(" ",
                F.trim(split_col.getItem(5)),
                F.trim(split_col.getItem(6))
               ).alias("datetime_str")
)

# Final guard: drop any row where latitude is still null after safe_double
data_df = data_df.filter(F.col("latitude").isNotNull())

total_data_lines = data_df.count()
print(f"Lines discarded (headers + malformed): {total_raw_lines - total_data_lines:,}")
print(f"GPS data lines parsed successfully   : {total_data_lines:,}")
print("User Guide v1.3 total (full dataset) : 24,876,978 points")
data_df.show(5)

Lines discarded (headers + malformed): 112,020
GPS data lines parsed successfully   : 24,876,978
User Guide v1.3 total (full dataset) : 24,876,978 points
+-------+--------------+---------+----------+-----------+-------------------+
|user_id| trajectory_id| latitude| longitude|altitude_ft|       datetime_str|
+-------+--------------+---------+----------+-----------+-------------------+
|    010|20081219114010|39.991376| 116.32641|      354.0|2008-12-19 11:40:10|
|    010|20081219114010|39.991358|116.326438|      361.0|2008-12-19 11:40:11|
|    010|20081219114010|39.991364|116.326458|      371.0|2008-12-19 11:40:12|
|    010|20081219114010|39.991391|116.326418|      322.0|2008-12-19 11:40:16|
|    010|20081219114010|39.991384|116.326436|      325.0|2008-12-19 11:40:17|
+-------+--------------+---------+----------+-----------+-------------------+
only showing top 5 rows


## Step 2: Type Casting & Unit Conversion
- Timestamp: parsed from `yyyy-MM-dd HH:mm:ss`. All times are GMT (User Guide s4.1).
- Altitude: convert feet to metres, replace -777 invalid sentinel with null.
  User Guide explicitly states -777 means the altitude reading is not valid.

In [8]:
# Parse timestamp - all times are GMT per User Guide Section 4.1
data_df = data_df.withColumn(
    "timestamp",
    F.to_timestamp("datetime_str", "yyyy-MM-dd HH:mm:ss")
).drop("datetime_str")

# Convert altitude feet -> metres.
# -777 is the explicit 'not valid' sentinel per User Guide Section 4.1.
# Replace with null rather than a numeric value that would skew statistics.
data_df = data_df.withColumn(
    "altitude_m",
    F.when(F.col("altitude_ft") == -777, None)
     .otherwise(F.round(F.col("altitude_ft") * 0.3048, 2))
).drop("altitude_ft")

data_df.cache()
data_df.printSchema()
data_df.show(5)

root
 |-- user_id: string (nullable = false)
 |-- trajectory_id: string (nullable = false)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- altitude_m: double (nullable = true)



+-------+--------------+---------+----------+-------------------+----------+
|user_id| trajectory_id| latitude| longitude|          timestamp|altitude_m|
+-------+--------------+---------+----------+-------------------+----------+
|    010|20081219114010|39.991376| 116.32641|2008-12-19 11:40:10|     107.9|
|    010|20081219114010|39.991358|116.326438|2008-12-19 11:40:11|    110.03|
|    010|20081219114010|39.991364|116.326458|2008-12-19 11:40:12|    113.08|
|    010|20081219114010|39.991391|116.326418|2008-12-19 11:40:16|     98.15|
|    010|20081219114010|39.991384|116.326436|2008-12-19 11:40:17|     99.06|
+-------+--------------+---------+----------+-------------------+----------+
only showing top 5 rows


## Step 3: Data Quality Checks

Checks performed:
1. Null counts on critical fields
2. Coordinates outside physically valid GPS ranges
3. Coordinate range summary - User Guide confirms Beijing is the primary
   region (lat ~39.4-41.1, lon ~115.4-117.6) with some overseas points
4. Trajectory length distribution

In [9]:
total_points = data_df.count()

# 1. Null counts
print("=== Null value counts ===")
data_df.select(
    F.count(F.when(F.col("latitude").isNull(),   1)).alias("null_latitude"),
    F.count(F.when(F.col("longitude").isNull(),  1)).alias("null_longitude"),
    F.count(F.when(F.col("timestamp").isNull(),  1)).alias("null_timestamp"),
    F.count(F.when(F.col("altitude_m").isNull(), 1)).alias("null_altitude_m"),
).show()

# 2. Invalid GPS ranges
invalid_coords = data_df.filter(
    (F.col("latitude")  < -90)  | (F.col("latitude")  > 90) |
    (F.col("longitude") < -180) | (F.col("longitude") > 180)
).count()
print(f"Points with invalid coordinates: {invalid_coords:,}")

# 3. Coordinate range
print("\n=== Coordinate range ===")
data_df.select(
    F.min("latitude").alias("lat_min"),
    F.max("latitude").alias("lat_max"),
    F.min("longitude").alias("lon_min"),
    F.max("longitude").alias("lon_max")
).show()

# 4. Trajectory length distribution
print("=== Points per trajectory ===")
traj_lengths_raw = data_df.groupBy("user_id", "trajectory_id") \
                           .agg(F.count("*").alias("pt_count"))
traj_lengths_raw.select(
    F.count("pt_count").alias("num_trajectories"),
    F.min("pt_count").alias("min_pts"),
    F.percentile_approx("pt_count", 0.25).alias("q1_pts"),
    F.percentile_approx("pt_count", 0.50).alias("median_pts"),
    F.percentile_approx("pt_count", 0.75).alias("q3_pts"),
    F.max("pt_count").alias("max_pts")
).show()

=== Null value counts ===


+-------------+--------------+--------------+---------------+
|null_latitude|null_longitude|null_timestamp|null_altitude_m|
+-------------+--------------+--------------+---------------+
|            0|             0|             0|          55484|
+-------------+--------------+--------------+---------------+



Points with invalid coordinates: 1

=== Coordinate range ===


+--------+----------------+------------+-----------+
| lat_min|         lat_max|     lon_min|    lon_max|
+--------+----------------+------------+-----------+
|1.044024|400.166666666667|-179.9695933|179.9969416|
+--------+----------------+------------+-----------+

=== Points per trajectory ===


+----------------+-------+------+----------+------+-------+
|num_trajectories|min_pts|q1_pts|median_pts|q3_pts|max_pts|
+----------------+-------+------+----------+------+-------+
|           18670|      3|   200|       506|  1367|  92645|
+----------------+-------+------+----------+------+-------+



## Step 4: Cleaning the Pipeline
Steps applied in order — each step's removal count is logged:
1. Drop rows with null latitude, longitude, or timestamp
2. Remove physically impossible GPS coordinates
3. Optional Beijing bounding box filter (controlled by `FILTER_TO_BEIJING`)
4. Remove trajectories shorter than `MIN_POINTS` (too sparse for TRACLUS segmentation)
5. Deduplicate identical (user_id, trajectory_id, timestamp) rows

In [10]:
clean_df = data_df

# Step 1 - Drop nulls on critical fields
clean_df = clean_df.dropna(subset=["latitude", "longitude", "timestamp"])
after_null = clean_df.count()
print(f"After null drop          : {after_null:,}  (removed {total_points - after_null:,})")

# Step 2 - Remove invalid coordinate ranges
clean_df = clean_df.filter(
    (F.col("latitude")  >= -90)  & (F.col("latitude")  <= 90) &
    (F.col("longitude") >= -180) & (F.col("longitude") <= 180)
)
after_coords = clean_df.count()
print(f"After coord range filter : {after_coords:,}  (removed {after_null - after_coords:,})")

# Step 3 - Optional Beijing bounding box filter
# User Guide: data spans 30+ cities and some overseas locations.
# Enabling this gives a geographically coherent clustering study but
# reduces coverage - document this decision in the report if enabled.
if FILTER_TO_BEIJING:
    clean_df = clean_df.filter(
        (F.col("latitude")  >= BEIJING_LAT_MIN) & (F.col("latitude")  <= BEIJING_LAT_MAX) &
        (F.col("longitude") >= BEIJING_LON_MIN) & (F.col("longitude") <= BEIJING_LON_MAX)
    )
    after_bbox = clean_df.count()
    print(f"After Beijing bbox filter: {after_bbox:,}  (removed {after_coords - after_bbox:,})")

# Step 4 - Remove short trajectories
traj_lengths = clean_df.groupBy("user_id", "trajectory_id") \
                        .agg(F.count("*").alias("pt_count"))
valid_trajs = traj_lengths.filter(F.col("pt_count") >= MIN_POINTS) \
                           .select("user_id", "trajectory_id")
clean_df = clean_df.join(valid_trajs, on=["user_id", "trajectory_id"], how="inner")
after_short = clean_df.count()
print(f"After short traj. filter : {after_short:,}  (removed {after_coords - after_short:,})")

# Step 5 - Deduplicate: same trajectory + same timestamp = duplicate GPS ping
clean_df = clean_df.dropDuplicates(["user_id", "trajectory_id", "timestamp"])
after_dedup = clean_df.count()
clean_df.cache()

print(f"After deduplication      : {after_dedup:,}  (removed {after_short - after_dedup:,})")
print(f"\nTotal removed: {total_points - after_dedup:,} ({(total_points - after_dedup)/total_points*100:.2f}%)")

After null drop          : 24,876,978  (removed 0)


After coord range filter : 24,876,977  (removed 1)


After short traj. filter : 24,875,256  (removed 1,721)


After deduplication      : 24,176,361  (removed 698,895)

Total removed: 700,617 (2.82%)


## Step 5: Feature Engineering

| Feature | Used by | Purpose |
|---|---|---|
| `point_order` | TRACLUS | Sequential index within trajectory for MDL segmentation |
| `time_delta_s` | All | Time gap between consecutive GPS points |
| `dist_to_prev_m` | All | Haversine distance to previous point |
| `speed_mps` | Artefact filter, K-medoids | Estimated instantaneous speed |

The **Haversine formula** computes the great-circle distance in metres between
two lat/lon points. Implemented as a PySpark UDF since Spark has no native function.

All lag-based computations use a window partitioned by `(user_id, trajectory_id)`
and ordered by `timestamp` - this is the correct distributed pattern as each
Spark worker handles its own partition of trajectories independently.

In [11]:
@udf(returnType=DoubleType())
def haversine_m(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres between two lat/lon points (Haversine formula)."""
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6_371_000.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Window: ordered by timestamp within each (user, trajectory) pair
traj_window = Window.partitionBy("user_id", "trajectory_id").orderBy("timestamp")

# Sequential point index - needed for TRACLUS MDL segmentation
clean_df = clean_df.withColumn("point_order", F.row_number().over(traj_window))

# Lag columns: previous point coordinates and timestamp
clean_df = clean_df \
    .withColumn("prev_lat", F.lag("latitude",  1).over(traj_window)) \
    .withColumn("prev_lon", F.lag("longitude", 1).over(traj_window)) \
    .withColumn("prev_ts",  F.lag("timestamp", 1).over(traj_window))

# Time delta in seconds between consecutive points
clean_df = clean_df.withColumn(
    "time_delta_s",
    (F.unix_timestamp("timestamp") - F.unix_timestamp("prev_ts")).cast(DoubleType())
)

# Haversine distance to previous point in metres
clean_df = clean_df.withColumn(
    "dist_to_prev_m",
    haversine_m(
        F.col("prev_lat"), F.col("prev_lon"),
        F.col("latitude"),  F.col("longitude")
    )
)

# Instantaneous speed in m/s
clean_df = clean_df.withColumn(
    "speed_mps",
    F.when(
        (F.col("time_delta_s") > 0) & F.col("dist_to_prev_m").isNotNull(),
        F.col("dist_to_prev_m") / F.col("time_delta_s")
    ).otherwise(None)
)

# Drop intermediate lag columns
clean_df = clean_df.drop("prev_lat", "prev_lon", "prev_ts")

clean_df.cache()
clean_df.printSchema()
clean_df.show(5)

root
 |-- user_id: string (nullable = false)
 |-- trajectory_id: string (nullable = false)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- altitude_m: double (nullable = true)
 |-- point_order: integer (nullable = false)
 |-- time_delta_s: double (nullable = true)
 |-- dist_to_prev_m: double (nullable = true)
 |-- speed_mps: double (nullable = true)



+-------+--------------+---------+----------+-------------------+----------+-----------+------------+------------------+------------------+
|user_id| trajectory_id| latitude| longitude|          timestamp|altitude_m|point_order|time_delta_s|    dist_to_prev_m|         speed_mps|
+-------+--------------+---------+----------+-------------------+----------+-----------+------------+------------------+------------------+
|    004|20090524004425|40.010685|116.321649|2009-05-24 00:44:25|     93.88|          1|        NULL|              NULL|              NULL|
|    004|20090524004425|40.010719|116.321711|2009-05-24 00:44:30|     94.79|          2|         5.0|6.4942451206566565|1.2988490241313313|
|    004|20090524004425| 40.01072|116.321733|2009-05-24 00:44:35|     98.15|          3|         5.0|1.8769679900870733|0.3753935980174147|
|    004|20090524004425|  40.0108|116.321742|2009-05-24 00:44:40|     98.45|          4|         5.0| 8.928556405433321|1.7857112810866642|
|    004|20090524004

### Removing GPS Artefacts via Speed Filter

GPS loggers occasionally produce teleportation artefacts - a point several kilometres
from the previous one with only a 1-second gap.

Threshold justification using User Guide transport mode data:
- Walk: ~1.4 m/s, Bike: ~4 m/s, Bus/Car: ~15-30 m/s, Train: ~50-80 m/s
- Airplane: ~200-250 m/s (cruising) - the fastest legitimate mode in this dataset
- 300 m/s (~1080 km/h) is well above any real transport mode while safely
  removing impossible GPS jumps

First points in each trajectory have speed_mps = null and are kept.

In [12]:
before_speed = clean_df.count()
clean_df = clean_df.filter(
    F.col("speed_mps").isNull() | (F.col("speed_mps") <= MAX_SPEED_MPS)
)
after_speed = clean_df.count()
print(f"Artefact points removed (speed > {MAX_SPEED_MPS} m/s): "
      f"{before_speed - after_speed:,} ({(before_speed - after_speed)/before_speed*100:.3f}%)")

# Speed distribution by transport mode - useful for report EDA section
print("\n=== Speed distribution (m/s) per transport mode ===")
clean_df.filter(F.col("speed_mps").isNotNull()).select(
    F.percentile_approx("speed_mps", 0.50).alias("median_mps"),
    F.percentile_approx("speed_mps", 0.90).alias("p90_mps"),
    F.percentile_approx("speed_mps", 0.99).alias("p99_mps"),
    F.max("speed_mps").alias("max_mps")
).show()

Artefact points removed (speed > 300.0 m/s): 1,127 (0.005%)

=== Speed distribution (m/s) per transport mode ===


+-----------------+------------------+----------------+------------------+
|       median_mps|           p90_mps|         p99_mps|           max_mps|
+-----------------+------------------+----------------+------------------+
|3.787875000128944|20.903135061818524|41.5238506294745|299.99157358212125|
+-----------------+------------------+----------------+------------------+



## Step 6: Transport Label Joining

From User Guide Section 4.2:
- 73 of 182 users have a labels.txt file
- Format is TAB-separated with a header row to skip
- Timestamp format uses slashes: `yyyy/MM/dd HH:mm:ss` (different from .plt files)
- All times are GMT

Label normalisation applied per User Guide guidance:
- `taxi` and `car` -> `driving` (User Guide: 'regard taxi and car as driving')
- All other modes kept as-is
- `train` / `subway` ambiguity noted but NOT merged here - left for analysis
  (User Guide: users may label Beijing light rail as either; can differentiate by distance)

Points not covered by any label interval -> `transport_mode = 'unlabelled'`

In [13]:
pre_label_count = clean_df.count()

label_rows = []
users_with_labels = 0

for user in selected_users:
    label_path = os.path.join(DATA_ROOT, user, "labels.txt")
    if os.path.exists(label_path):
        users_with_labels += 1
        with open(label_path, "r") as f:
            for i, line in enumerate(f):
                if i == 0:   # skip header row
                    continue
                # TAB-separated per User Guide Section 4.2
                parts = line.strip().split("\t")
                if len(parts) == 3:
                    start_t, end_t, mode = parts
                    mode = mode.lower().strip()
                    # Normalise: taxi and car both mean driving (User Guide s4.2)
                    if mode in ("taxi", "car"):
                        mode = "driving"
                    label_rows.append((start_t.strip(), end_t.strip(), mode, user))

print(f"Users with labels.txt : {users_with_labels}")
print(f"  (User Guide states 73 for full dataset)")
print(f"Label intervals loaded: {len(label_rows):,}")

if label_rows:
    label_schema = StructType([
        StructField("start_str",      StringType()),
        StructField("end_str",        StringType()),
        StructField("transport_mode", StringType()),
        StructField("user_id",        StringType()),
    ])
    labels_df = spark.createDataFrame(label_rows, schema=label_schema)

    # Note: labels.txt timestamps use slashes yyyy/MM/dd HH:mm:ss
    # This differs from the .plt datetime format (User Guide s4.2)
    labels_df = labels_df \
        .withColumn("label_start",
                    F.to_timestamp("start_str", "yyyy/MM/dd HH:mm:ss")) \
        .withColumn("label_end",
                    F.to_timestamp("end_str",   "yyyy/MM/dd HH:mm:ss")) \
        .drop("start_str", "end_str")

    # Join on user_id AND timestamp falling within [label_start, label_end]
    clean_df = clean_df.join(
        labels_df,
        on=(
            (clean_df.user_id   == labels_df.user_id) &
            (clean_df.timestamp >= labels_df.label_start) &
            (clean_df.timestamp <= labels_df.label_end)
        ),
        how="left"
    ).drop(labels_df.user_id, "label_start", "label_end")

    clean_df = clean_df.fillna({"transport_mode": "unlabelled"})

else:
    # Subset may not include any labelled users
    clean_df = clean_df.withColumn("transport_mode", F.lit("unlabelled"))
    print("No label files in this subset - all points marked 'unlabelled'")

clean_df.cache()
print("\n=== Transport mode distribution ===")
clean_df.groupBy("transport_mode").count().orderBy(F.desc("count")).show()

# Note for report - User Guide Section 4.2:
# 'train' and 'subway' may refer to the same Beijing light rail lines.
# Real inter-city trains can be differentiated from light rail by total trajectory distance.
print("NOTE: train/subway ambiguity documented in User Guide s4.2.")
print("Consider merging or differentiating by trajectory distance in analysis.")

Users with labels.txt : 69
  (User Guide states 73 for full dataset)
Label intervals loaded: 14,718

=== Transport mode distribution ===


+--------------+--------+
|transport_mode|   count|
+--------------+--------+
|    unlabelled|19310042|
|          walk| 1365259|
|           bus| 1207279|
|          bike|  784895|
|       driving|  720547|
|         train|  560923|
|        subway|  277117|
|      airplane|    9193|
|          boat|    3566|
|           run|    1967|
|    motorcycle|     338|
+--------------+--------+

NOTE: train/subway ambiguity documented in User Guide s4.2.
Consider merging or differentiating by trajectory distance in analysis.


## Step 7: Outputs

### Trajectory-Level Summary DataFrame

Aggregates each trajectory to a single row. This is the primary input for
K-medoids (whole-trajectory clustering) and for EDA.

Distance distribution from User Guide Figure 2 (used as a sanity check):
- 36% of trajectories < 5 km
- 36% between 5-20 km
- 23% between 20-100 km
- 5% >= 100 km

In [14]:
traj_summary = clean_df.groupBy("user_id", "trajectory_id").agg(
    F.count("*").alias("num_points"),
    F.min("timestamp").alias("start_time"),
    F.max("timestamp").alias("end_time"),
    # Total trajectory distance in km
    F.round(F.sum("dist_to_prev_m") / 1000.0, 3).alias("total_dist_km"),
    F.round(F.avg("speed_mps"), 3).alias("avg_speed_mps"),
    F.round(F.max("speed_mps"), 3).alias("max_speed_mps"),
    # Bounding box
    F.min("latitude").alias("lat_min"),
    F.max("latitude").alias("lat_max"),
    F.min("longitude").alias("lon_min"),
    F.max("longitude").alias("lon_max"),
    # Start and end coordinates (for whole-trajectory distance measures)
    F.first("latitude",  ignorenulls=True).alias("start_lat"),
    F.first("longitude", ignorenulls=True).alias("start_lon"),
    F.last("latitude",   ignorenulls=True).alias("end_lat"),
    F.last("longitude",  ignorenulls=True).alias("end_lon"),
    # Duration in seconds
    (
        F.unix_timestamp(F.max("timestamp")) -
        F.unix_timestamp(F.min("timestamp"))
    ).alias("duration_s"),
    # Most frequent transport mode in this trajectory
    F.first("transport_mode", ignorenulls=True).alias("primary_mode")
)

num_trajs = traj_summary.count()
print(f"Trajectory summary: {num_trajs:,} rows")
traj_summary.show(5)

# Sanity check: compare distance distribution against User Guide Figure 2
print("\n=== Distance distribution vs User Guide Figure 2 ===")
traj_summary.select(
    F.count(F.when(F.col("total_dist_km") < 5,   1)).alias("lt_5km"),
    F.count(F.when((F.col("total_dist_km") >= 5)  & (F.col("total_dist_km") < 20),  1)).alias("5_to_20km"),
    F.count(F.when((F.col("total_dist_km") >= 20) & (F.col("total_dist_km") < 100), 1)).alias("20_to_100km"),
    F.count(F.when(F.col("total_dist_km") >= 100, 1)).alias("gte_100km"),
    F.count("*").alias("total")
).show()
print("User Guide expected (full dataset): 36% <5km | 36% 5-20km | 23% 20-100km | 5% >=100km")

Trajectory summary: 18,364 rows
+-------+--------------+----------+-------------------+-------------------+-------------+-------------+-------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------+------------+
|user_id| trajectory_id|num_points|         start_time|           end_time|total_dist_km|avg_speed_mps|max_speed_mps|         lat_min|         lat_max|         lon_min|         lon_max|       start_lat|       start_lon|         end_lat|         end_lon|duration_s|primary_mode|
+-------+--------------+----------+-------------------+-------------------+-------------+-------------+-------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------------+----------+------------+
|    125|20080509081231|        87|2008-05-09 08:12:31|2008-05-09 10:31:01|        3.845|        3.242|       15.397|39.9765333333333|

+------+---------+-----------+---------+-----+
|lt_5km|5_to_20km|20_to_100km|gte_100km|total|
+------+---------+-----------+---------+-----+
|  6490|     6713|       4300|      861|18364|
+------+---------+-----------+---------+-----+

User Guide expected (full dataset): 36% <5km | 36% 5-20km | 23% 20-100km | 5% >=100km


### Reproducible Development Subset

Documented per project brief requirement: all subset creation must be recorded.

Sampling at trajectory level (not point level) preserves complete trajectories,
which is required for TRACLUS and K-medoids. The fraction is configurable.

In [15]:
DEV_FRACTION = 0.10   # 10% of trajectories for development iteration

dev_traj_ids = traj_summary \
    .select("user_id", "trajectory_id") \
    .sample(fraction=DEV_FRACTION, seed=RANDOM_SEED)

dev_df = clean_df.join(dev_traj_ids, on=["user_id", "trajectory_id"], how="inner")

print(f"Development subset  : {dev_traj_ids.count():,} trajectories | {dev_df.count():,} points")
print(f"Full cleaned dataset: {num_trajs:,} trajectories | {clean_df.count():,} points")

Development subset  : 1,853 trajectories | 2,473,386 points
Full cleaned dataset: 18,364 trajectories | 24,241,126 points


### Save Processed Data as Parquet

Parquet preserves schema, supports predicate pushdown, and is column-oriented
for efficient field-level reads in downstream PySpark algorithm notebooks.

In [16]:
clean_df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/geolife_clean_points")
traj_summary.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/geolife_traj_summary")
dev_df.write.mode("overwrite").parquet(f"{OUTPUT_DIR}/geolife_dev_subset")

print("Saved:")
print(f"  {OUTPUT_DIR}/geolife_clean_points   (point-level, all algorithms)")
print(f"  {OUTPUT_DIR}/geolife_traj_summary   (trajectory-level, K-medoids input)")
print(f"  {OUTPUT_DIR}/geolife_dev_subset     (10% point-level subset for development)")

Saved:
  /Users/abhinavtadi/Desktop/University of Nottingham/Sem 2/Big Data Learning and Technologies/Big_Data_Group_5/Clean_Dataset/geolife_clean_points   (point-level, all algorithms)
  /Users/abhinavtadi/Desktop/University of Nottingham/Sem 2/Big Data Learning and Technologies/Big_Data_Group_5/Clean_Dataset/geolife_traj_summary   (trajectory-level, K-medoids input)
  /Users/abhinavtadi/Desktop/University of Nottingham/Sem 2/Big Data Learning and Technologies/Big_Data_Group_5/Clean_Dataset/geolife_dev_subset     (10% point-level subset for development)


### Final Summary

In [17]:
final_points = clean_df.count()
final_trajs  = traj_summary.count()
final_users  = clean_df.select("user_id").distinct().count()

print("=" * 56)
print("            PREPROCESSING SUMMARY")
print("=" * 56)
print(f"  Users loaded          : {final_users}")
print(f"  Trajectories          : {final_trajs:,}")
print(f"  GPS points            : {final_points:,}")
print(f"  Points removed        : {total_points - pre_label_count:,}"
      f"  ({(total_points - pre_label_count)/total_points*100:.2f}%)")
print("-" * 56)
print(f"  USE_SUBSET            : {USE_SUBSET}")
print(f"  FILTER_TO_BEIJING     : {FILTER_TO_BEIJING}")
print(f"  MIN_POINTS            : {MIN_POINTS}")
print(f"  MAX_SPEED_MPS         : {MAX_SPEED_MPS}")
print(f"  RANDOM_SEED           : {RANDOM_SEED}")
print("=" * 56)
print("\nFinal schema:")
clean_df.printSchema()

            PREPROCESSING SUMMARY
  Users loaded          : 181
  Trajectories          : 18,364
  GPS points            : 24,241,126
  Points removed        : 701,744  (2.82%)
--------------------------------------------------------
  USE_SUBSET            : False
  FILTER_TO_BEIJING     : False
  MIN_POINTS            : 10
  MAX_SPEED_MPS         : 300.0
  RANDOM_SEED           : 42

Final schema:
root
 |-- user_id: string (nullable = false)
 |-- trajectory_id: string (nullable = false)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timestamp: timestamp (nullable = false)
 |-- altitude_m: double (nullable = true)
 |-- point_order: integer (nullable = false)
 |-- time_delta_s: double (nullable = true)
 |-- dist_to_prev_m: double (nullable = true)
 |-- speed_mps: double (nullable = true)
 |-- transport_mode: string (nullable = false)



### Clearing Cache

Any data/elements currently in cache will no longer be used. It is better to clear the cache now, so we can save resources when caching further data. This will also prevent or decrease the Out of Memory problems significantly.

In [18]:
spark.catalog.clearCache()
print("Cache Cleared")

Cache Cleared


---
## Section 3: Common Tools
---

---
## Section 4: Methodologies
--- 

---
## Section 5: End
--- 

In [19]:
spark.stop()